In [ ]:
BATCH_SIZE = 512
EPOCH_SCRATCH = 8
EPOCH_TRANSFER = 8
EPOCH_FINETUNE = 8

In [ ]:
# tracking the vram usage accross the process

_peak_vram_used_mb = 0.0

def log_gpu_memory(label, tf=None):
    """
    Log current VRAM state (via nvidia-smi) and TF own allocator stats.
    Pass the tf module to also get TF-side peak allocation tracking.
    """
    global _peak_vram_used_mb

    # nvidia-smi: OS-level view of VRAM
    try:
        smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10
        )
        if smi.returncode == 0:
            used_str, free_str, total_str = [p.strip() for p in smi.stdout.strip().split(",")]
            print(f"  VRAM [{label}]")
            print(f"    nvidia-smi  used={used_str}  free={free_str}  total={total_str}")

            # track peak across all snapshots
            used_mb = float(used_str.replace("MiB", "").strip())
            if used_mb > _peak_vram_used_mb:
                _peak_vram_used_mb = used_mb
    except Exception:
        pass

    # TF allocator: TF-side view of what it allocated
    if tf is not None:
        try:
            gpus = tf.config.list_physical_devices("GPU")
            if gpus:
                mem = tf.config.experimental.get_memory_info("GPU:0")
                current_mb = mem["current"] / 1024**2
                peak_mb    = mem["peak"]    / 1024**2
                print(f"    TF allocator  current={current_mb:.1f} MiB  peak={peak_mb:.1f} MiB")
        except Exception:
            pass

In [ ]:
import platform, os, subprocess, sys, datetime, psutil

SEP  = "=" * 60
SEP2 = "-" * 60

def _smi(query):
    try:
        r = subprocess.run(
            ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10
        )
        return [l.strip() for l in r.stdout.strip().splitlines()] if r.returncode == 0 else []
    except Exception:
        return []

print(SEP)
print("  NOTEBOOK SYSTEM INFO")
print(f"  {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(SEP)

# ── Host ──────────────────────────────────────────────────────
print("\n  HOST")
print(SEP2)
print(f"  Hostname       : {platform.node()}")
print(f"  OS             : {platform.system()} {platform.release()} ({platform.version()[:60]})")
print(f"  Architecture   : {platform.machine()}")
print(f"  Python         : {sys.version.split()[0]}  ({sys.executable})")
print(f"  Working dir    : {os.getcwd()}")
print(f"  PBS Job ID     : {os.environ.get('PBS_JOBID',  'N/A (local/interactive)')}")
print(f"  PBS Queue      : {os.environ.get('PBS_QUEUE',  'N/A')}")

# ── CPU ───────────────────────────────────────────────────────
print("\n  CPU")
print(SEP2)
try:
    model = next(
        (l.split(":")[1].strip() for l in open("/proc/cpuinfo").readlines() if "model name" in l),
        "unavailable"
    )
    print(f"  Model          : {model}")
except Exception:
    print("  Model          : unavailable")
print(f"  Physical cores : {psutil.cpu_count(logical=False)}")
print(f"  Logical cores  : {psutil.cpu_count(logical=True)}")
print(f"  Current freq   : {psutil.cpu_freq().current:.0f} MHz" if psutil.cpu_freq() else "  Current freq   : unavailable")

# ── RAM ───────────────────────────────────────────────────────
print("\n  RAM")
print(SEP2)
vm = psutil.virtual_memory()
print(f"  Total          : {vm.total    / 1024**3:.2f} GB")
print(f"  Available      : {vm.available / 1024**3:.2f} GB")
print(f"  Used           : {vm.used     / 1024**3:.2f} GB  ({vm.percent:.1f}%)")

# ── Disk ──────────────────────────────────────────────────────
print("\n  DISK  (working directory partition)")
print(SEP2)
try:
    du = psutil.disk_usage(os.getcwd())
    print(f"  Total          : {du.total / 1024**3:.1f} GB")
    print(f"  Used           : {du.used  / 1024**3:.1f} GB  ({du.percent:.1f}%)")
    print(f"  Free           : {du.free  / 1024**3:.1f} GB")
except Exception:
    print("  unavailable")

# ── GPU ───────────────────────────────────────────────────────
print("\n  GPU")
print(SEP2)
names    = _smi("name")
drivers  = _smi("driver_version")
mem_tot  = _smi("memory.total")
mem_free = _smi("memory.free")
temps    = _smi("temperature.gpu")
utils    = _smi("utilization.gpu")
caps     = _smi("compute_cap")

if names:
    for i, name in enumerate(names):
        print(f"  GPU {i}          : {name}")
        if i < len(drivers):  print(f"    Driver       : {drivers[i]}")
        if i < len(caps):     print(f"    Compute cap  : {caps[i]}")
        if i < len(mem_tot):  print(f"    VRAM total   : {mem_tot[i]} MiB")
        if i < len(mem_free): print(f"    VRAM free    : {mem_free[i]} MiB")
        if i < len(temps):    print(f"    Temperature  : {temps[i]} °C")
        if i < len(utils):    print(f"    GPU util     : {utils[i]} %")
else:
    print("  No GPU detected (nvidia-smi unavailable or no GPU present)")

# ── Key packages ──────────────────────────────────────────────
print("\n  KEY PACKAGES")
print(SEP2)
packages = ["tensorflow", "keras", "numpy", "pandas", "matplotlib",
            "sklearn", "scipy", "torch", "cv2", "PIL"]
for pkg in packages:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "installed")
        print(f"  {pkg:<20}: {ver}")
    except ImportError:
        pass   # silently skip packages that aren't installed

print()
print(SEP)
print("  System info complete")
print(SEP)

# keep a reference for the footer cell
import time as _time
_NOTEBOOK_START_TIME = _time.time()
_NOTEBOOK_START_DT   = datetime.datetime.now()

In [ ]:
from datetime import datetime

start_time = datetime.now()
TIMESTAMP = start_time.strftime("%Y%m%d_%H%M%S")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

In [ ]:
# Adjust path to the downloaded dataset structure
with open('./dataset_path.txt', 'r') as file:
    file_content = file.read()

DATA_DIR = file_content + "/chest_xray"
IMG_SIZE = (224, 224)
SEED = 42

# the dataset already has train/val/test splits, so load them directly
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR + "/train",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR + "/val",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Classes: {class_names}, Nombre de classes: {num_classes}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
scratch_model = models.Sequential([
    layers.Input(shape=(224,224,3)),
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

In [ ]:
scratch_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

log_gpu_memory("before scratch training", tf)

history_scratch = scratch_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCH_SCRATCH
)
scratch_model.save(f"models/{TIMESTAMP}_scratch_model.keras")
log_gpu_memory("after fine-tuning", tf)

In [ ]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False

In [ ]:
inputs = layers.Input(shape=(224,224,3))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

transfer_model = models.Model(inputs, outputs)

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

log_gpu_memory("before transfer training", tf)

history_transfer = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCH_TRANSFER
)
transfer_model.save(f"models/{TIMESTAMP}_transfer_model.keras")
log_gpu_memory("after transfer training", tf)

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-4]:
    layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


log_gpu_memory("before fine-tuning", tf)
history_finetune = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCH_FINETUNE
)
transfer_model.save(f"models/{TIMESTAMP}_finetuned_model.keras")
log_gpu_memory("after fine-tuning", tf)

In [ ]:
end_time = datetime.now()
execution_time = end_time - start_time

print(f"Execution time: {execution_time}")
print(f"  Peak VRAM used     : {_peak_vram_used_mb:.0f} MiB  ({_peak_vram_used_mb/1024:.2f} GiB)")

In [ ]:
import psutil
import os
import time

def get_resource_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    cpu_percent = process.cpu_percent(interval=1.0)
    stats = {
        "Memory Used (MB)": round(mem_info.rss / (1024 ** 2), 2),
        "Memory Percent": process.memory_percent(),
        "CPU Percent": cpu_percent,
        "Total Runtime (s)": round(time.time() - start_time, 2),
    }
    return stats

# Record the start time at the beginning of the notebook
start_time = time.time()

# At the end of the notebook, print resource usage
resource_usage = get_resource_usage()
print("\nResource Usage Statistics:")
for key, value in resource_usage.items():
    print(f"{key:20}: {value}")
